In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import warnings

# Use current directory as root for scanning
current_dir = os.getcwd() # Or specify explicitly if needed
print(f"Working directory: {current_dir}")

## 2. 解析 PQR 文件结构

定义 `parse_pqr` 函数。该函数读取 PQR 文件，按行拆分数据，提取坐标 (x, y, z)、电荷 (Charge)、半径 (Radius) 以及原子名称（第3列）和残基编号（第6列）等关键信息，并将其转换为 DataFrame 格式。

In [ ]:
def parse_pqr(pqr_file):
    """
    Parses a PQR file to extract coordinates, charges, radii, atom names and residue info.
    
    Args:
        pqr_file (str): Path to the .pqr file.
        
    Returns:
        pd.DataFrame: DataFrame containing atom information.
    """
    atoms = []
    with open(pqr_file, 'r') as f:
        for line in f:
            if line.startswith('ATOM') or line.startswith('HETATM'):
                try:
                    parts = line.split()
                    
                    # Standard PQR parsing logic (from back for safety on variable chain columns)
                    # Field order usually: Record, Serial, AtomName, ResName, (Chain), ResSeq, X, Y, Z, Q, R
                    
                    radius = float(parts[-1])
                    charge = float(parts[-2])
                    z = float(parts[-3])
                    y = float(parts[-4])
                    x = float(parts[-5])
                    
                    # Extract specific columns for centroid identification
                    # 3rd column (index 2) -> Atom Name
                    # 6th column (index 5) -> Residue ID (assuming chain present) or possibly X coordinate if missing
                    # We store it as a string primarily to check against '167'
                    
                    atom_name = parts[2] if len(parts) > 2 else ""
                    
                    # The prompt specifies "6th column". In 0-indexed split list, this is index 5.
                    col6 = parts[5] if len(parts) > 5 else ""

                    atom_info = {
                        'x': x,
                        'y': y,
                        'z': z,
                        'charge': charge,
                        'radius': radius,
                        'atom_name': atom_name,
                        'col6': col6,
                        'line': line.strip()
                    }
                    atoms.append(atom_info)
                except (ValueError, IndexError):
                    continue
                    
    return pd.DataFrame(atoms)

## 3. 计算静电势 (Electrostatic Potential)

定义 `calculate_esp` 函数。利用 Coulomb 公式 ($V_i = \sum \frac{q_j}{r_{ij}}$) 和 Numpy 的向量化广播机制，计算每个原子受周围所有其他原子电荷影响的静电势总和。

In [ ]:
def calculate_esp(df):
    """
    Calculates the Electrostatic Potential (ESP) for each atom.
    ESP_i = Sum( q_j / d_ij ) for j != i
    
    Args:
        df (pd.DataFrame): DataFrame with x, y, z, charge.
        
    Returns:
        np.array: Array of ESP values.
    """
    coords = df[['x', 'y', 'z']].values
    charges = df['charge'].values
    n = len(df)
    esp = np.zeros(n)
    
    # Calculate pairwise distances efficiently
    delta = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    
    # Euclidean distances
    dists = np.sqrt(np.sum(delta**2, axis=-1))
    
    # Avoid division by zero on diagonal (self-interaction)
    # We treat self-potential as 0 for this interaction summation context (or handle infinite)
    with np.errstate(divide='ignore'):
        inv_dists = 1.0 / dists
    np.fill_diagonal(inv_dists, 0.0)
    
    # ESP calculation: Matrix vector multiplication
    # (N, N) * (N,) -> (N,)
    esp = inv_dists @ charges
    
    return esp

## 4. 计算到质心 (NZ-167) 的距离

定义 `calculate_centroid_distance` 函数。实现核心逻辑：在 DataFrame 中筛选原子名称为 'NZ' 且第6列（通常为残基ID）为 '167' 的原子作为质心。如果找到质心，计算所有其他原子到该质心的欧几里得距离；如果未找到，则返回 NaN。

In [ ]:
def calculate_centroid_distance(df):
    """
    Calculates the distance of each atom to a centroid atom using coordinates.
    Centroid condition: 3rd column (atom_name) == 'NZ' and 6th column (col6) == '167'.
    
    Args:
        df (pd.DataFrame): DataFrame with x, y, z, atom_name, col6.
        
    Returns:
        np.array: Array of distances.
    """
    # Filter for the centroid atom
    # Identifying column 3 as 'NZ' and column 6 as '167'
    # Note: col6 is a string from parsing
    centroid_atom = df[(df['atom_name'] == 'NZ') & (df['col6'] == '167')]
    
    if centroid_atom.empty:
        print("    Warning: Centroid atom (NZ, 167) not found. Setting distances to NaN.")
        return np.full(len(df), np.nan)
    
    # Use the first one found
    c_x = centroid_atom.iloc[0]['x']
    c_y = centroid_atom.iloc[0]['y']
    c_z = centroid_atom.iloc[0]['z']
    centroid_coords = np.array([c_x, c_y, c_z])
    
    atom_coords = df[['x', 'y', 'z']].values
    
    # Euclidean distance
    # sqrt(sum((x-c)^2))
    diff = atom_coords - centroid_coords
    dists = np.sqrt(np.sum(diff**2, axis=1))
    
    return dists

## 5. 批量处理文件并生成描述符矩阵

定义主处理流程。使用 `glob` 递归查找当前目录及子目录下的 `.pqr` 文件，依次调用上述函数计算 ESP 和质心距离，整合 `radius`, `esp`, `distance_to_centroid` 等特征，并将结果保存为 `_descriptor.csv` 文件。

In [ ]:
def process_directory(root_dir):
    print(f"Scanning {root_dir}...")
    pqr_files = glob.glob(os.path.join(root_dir, '**', '*.pqr'), recursive=True)
    
    print(f"Found {len(pqr_files)} PQR files.")
    
    for pqr_path in pqr_files:
        print(f"Processing {os.path.basename(pqr_path)}...")
        try:
            df = parse_pqr(pqr_path)
            if df.empty:
                print(f"  Warning: No valid atoms found in {pqr_path}")
                continue
                
            # Calculate ESP
            esp = calculate_esp(df)
            
            # Calculate Distance to Centroid (NZ 167)
            dists = calculate_centroid_distance(df)

            # Add to dataframe
            df['esp'] = esp
            df['distance_to_centroid'] = dists
            
            # Construct descriptor matrix
            # Selecting: Radius, ESP, and Distance
            descriptor_matrix = df[['radius', 'esp', 'distance_to_centroid']]
            
            # Save the matrix
            # Structure: original_name + _descriptor.csv
            output_path = pqr_path.replace('.pqr', '_descriptor.csv')
            
            # Ensure unique filename if .pqr is not extension (rare edge case)
            if output_path == pqr_path:
                 output_path += "_descriptor.csv"
                 
            descriptor_matrix.to_csv(output_path, index=False)
            print(f"  Saved descriptor to {output_path}")
            
        except Exception as e:
            print(f"  Error processing {pqr_path}: {e}")

# 执行处理 (默认处理当前 notebooks 所在目录)
# 也可以修改 path 为特定目录
target_path = os.getcwd() 
process_directory(target_path)

## 6. 模型训练数据准备

从指定的 CSV 文件加载产率数据，并匹配当前目录下生成的描述符文件。
将每个分子的原子级描述符聚合（均值、标准差、最大值、最小值）为固定长度的特征向量。


In [ ]:
import pandas as pd
import numpy as np
import glob
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

# --- Configuration ---
# 默认路径，可以根据需要修改
# 如果通过脚本运行，可以使用 sys.argv 或环境变量覆盖
YIELD_CSV_PATH = os.environ.get('YIELD_CSV_PATH', 'yield.csv') 

def load_data(yield_csv_path, descriptor_dir='.'):
    print(f"Loading yields from {yield_csv_path}...")
    try:
        yield_df = pd.read_csv(yield_csv_path)
    except FileNotFoundError:
        print("Yield file not found. Please set YIELD_CSV_PATH correctly.")
        return None, None, None

    X_list = []
    y_list = []
    names = []

    # 查找所有描述符文件
    descriptor_files = glob.glob(os.path.join(descriptor_dir, '*_descriptor.csv'))
    print(f"Found {len(descriptor_files)} descriptor files.")

    for desc_file in descriptor_files:
        filename = os.path.basename(desc_file)
        name_no_suffix = filename.replace('_descriptor.csv', '')
        
        # 匹配逻辑：检查产率表中的 Mutation 字段是否包含在文件名中
        match_row = None
        for _, row in yield_df.iterrows():
            mut = str(row['Mutation']).strip().lower()
            if not mut: continue
            
            # 简化匹配：如果文件名包含突变名称 (如 's18a' in '7p76_s18a_descriptor.csv')
            # 特殊处理 'wt'
            if f"_{mut}_" in filename or filename.endswith(f"_{mut}_descriptor.csv") or \
               (mut == 'wt' and '_wt_' in filename):
                match_row = row
                break
        
        if match_row is None:
            continue

        try:
            # 读取描述符
            desc_df = pd.read_csv(desc_file)
            if desc_df.empty: continue
            
            # --- 特征工程 ---
            # 聚合原子描述符为分子描述符
            # 针对 radius, esp, distance_to_centroid 计算统计量
            stats = []
            for col in ['radius', 'esp', 'distance_to_centroid']:
                if col in desc_df.columns:
                    col_data = desc_df[col].dropna()
                    if len(col_data) == 0:
                        stats.extend([0, 0, 0, 0])
                    else:
                        stats.extend([
                            col_data.mean(),
                            col_data.std(),
                            col_data.min(),
                            col_data.max()
                        ])
                else:
                    stats.extend([0, 0, 0, 0])
            
            X_list.append(stats)
            y_list.append(match_row['Yield'])
            names.append(match_row['Mutation'])
            
        except Exception as e:
            print(f"Error reading {desc_file}: {e}")

    return np.array(X_list), np.array(y_list), names

# 执行加载
X, y, sample_names = load_data(YIELD_CSV_PATH, target_path)

if X is not None and len(X) > 0:
    print(f"Data loaded. X shape: {X.shape}, y shape: {y.shape}")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
else:
    print("No data matched. Please check file naming conventions.")


In [ ]:
from sklearn.ensemble import RandomForestRegressor

if X is not None and len(X) > 0:
    print("=== Training Random Forest ===")
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    
    y_pred_rf = rf.predict(X_test)
    rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
    r2_rf = r2_score(y_test, y_pred_rf)
    
    print(f"Random Forest - RMSE: {rmse_rf:.4f}, R2: {r2_rf:.4f}")
    
    plt.figure(figsize=(5,5))
    plt.scatter(y_test, y_pred_rf, color='blue', label='Pred vs Actual')
    plt.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2)
    plt.xlabel('Actual Yield')
    plt.ylabel('Predicted Yield')
    plt.title('Random Forest')
    plt.legend()
    plt.show()


In [ ]:
from sklearn.cross_decomposition import PLSRegression

if X is not None and len(X) > 0:
    print("=== Training PLS Regression ===")
    # n_components needs to be < min(n_samples, n_features)
    n_comp = min(3, X.shape[1], len(X_train)-1)
    if n_comp < 1: n_comp = 1
    
    pls = PLSRegression(n_components=n_comp)
    pls.fit(X_train, y_train)
    
    y_pred_pls = pls.predict(X_test)
    rmse_pls = np.sqrt(mean_squared_error(y_test, y_pred_pls))
    r2_pls = r2_score(y_test, y_pred_pls)
    
    print(f"PLS - RMSE: {rmse_pls:.4f}, R2: {r2_pls:.4f}")
    
    plt.figure(figsize=(5,5))
    plt.scatter(y_test, y_pred_pls, color='green', label='Pred vs Actual')
    plt.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2)
    plt.xlabel('Actual Yield')
    plt.ylabel('Predicted Yield')
    plt.title('PLS Regression')
    plt.legend()
    plt.show()


In [ ]:
from sklearn.linear_model import Lasso

if X is not None and len(X) > 0:
    print("=== Training Lasso Regression ===")
    lasso = Lasso(alpha=0.1) # Alpha is a hyperparameter to tune
    lasso.fit(X_train, y_train)
    
    y_pred_lasso = lasso.predict(X_test)
    rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
    r2_lasso = r2_score(y_test, y_pred_lasso)
    
    print(f"Lasso - RMSE: {rmse_lasso:.4f}, R2: {r2_lasso:.4f}")
    
    plt.figure(figsize=(5,5))
    plt.scatter(y_test, y_pred_lasso, color='red', label='Pred vs Actual')
    plt.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2)
    plt.xlabel('Actual Yield')
    plt.ylabel('Predicted Yield')
    plt.title('Lasso Regression')
    plt.legend()
    plt.show()
